# Financial RAG

## Chunking

Chunking is done by parse_filing.py, then put into the data/ folder.

## Ingestion

In [ ]:
import json
import os
import sys
from pathlib import Path

import psycopg2
import voyageai
from llama_index.core.schema import TextNode
from llama_index.vector_stores.postgres import PGVectorStore

PARSED_DIR = Path("data/parsed")
COMPANIES = ["AAPL", "KO", "LUV"]

EMBED_DIM = 1024
EMBED_MODEL = "voyage-context-4"

PG_HOST = os.environ.get("PG_HOST", "localhost")
PG_PORT = int(os.environ.get("PG_PORT", "5432"))
PG_DATABASE = os.environ.get("PG_DATABASE", "postgres")
PG_USER = os.environ.get("PG_USER", "postgres")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "postgres")
PG_TABLE = "finance_filings"


In [ ]:
def load_records(company:  str) -> list[dict]:
  path = PARSED_DIR / f"{company}.json"
  return json.loads(path.read_text())

def embed_company(vo: voyageai.Client, records: list[dict]) -> list[list[float]]:
  """One contextualized embed call per company. All company chunks go in as onedocument"""
  chunk_texts = [r["text"] for r in records]
  result = vo.contextualized_embed(
    inputs=[chunk_texts],
    model=EMBED_MODEL,
    input_type="document",
    output_dimension=EMBED_DIM
  )

  return result.results[0].embeddings



In [ ]:
def delete_existing(company: str) -> None:
  """Thus function is for idempotency, to wipe the company existing rows before we add a fresh one"""
  conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DATABASE,
    user=PG_USER, password=PG_PASSWORD
  )

  try:
    with conn, conn.cursor() as cur:
      cur.execute(
        f"DELETE FROM data_{PG_TABLE} WHERE metadata_->>'company' = %s",
        (company,),
      )
      print(f" deleted {cur.rowcount} existing ")
  finally:
    conn.close()

In [ ]:
companies = ["AAPL"]

In [ ]:
for c in companies:
  if c not in COMPANIES:
    sys.exit(f"Unknown company {c!r}, expected one of {COMPANIES}")

vo = voyageai.Client()

vector_store = PGVectorStore.from_params(
  host = PG_HOST, port = str(PG_PORT), database = PG_DATABASE,
  user = PG_USER, password = PG_PASSWORD,
  table_name=PG_TABLE, embed_dim = EMBED_DIM,
)

for company in companies:
  print(f"Processing {company}...")
  records = load_records(company)
  print(f"  {len(records)} chunks loaded from {company}.json")

  embeddings = embed_company(vo, records)

  assert len(embeddings) == len(records), (
    f"Got {len(embeddings)} embeddings for {len(records)} chunks"
    "contextualized_embed response shape didn't match what was expected"
  )

  nodes = []

  for i, (record, embedding) in enumerate(zip(records, embeddings)):
    node = TextNode(
      id_=f"{company}_{i}",
      text=record["text"],
      embedding=embedding,
      metadata={
        "company": record["company"],
        "section": record["section"],
        "page": record["page"],
        "fiscal_period": record["fiscal_period"],
        "node_type": record["node_type"],
      }
    )
    nodes.append(node)

  delete_existing(company=company)
  vector_store.add(nodes)
  print(f"  inserted {len(nodes)} nodes for {company}")

print("Done.")